In [2]:
from splinter import Browser
from bs4 import BeautifulSoup as soup
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import random 
import pandas as pd
import os

In [3]:
service = Service(ChromeDriverManager().install()) 

In [4]:
class MovieScraper:
    def __init__(self, base_url):
        self.base_url = base_url
        self.results = []

    def _init_browser(self):
        from selenium.webdriver.chrome.options import Options
         
        chrome_options = Options()
         
        prefs = {"profile.managed_default_content_settings.images": 2}
        chrome_options.add_experimental_option("prefs", prefs) 
        
        return Browser('chrome', service=service, options=chrome_options, headless=False)
    
    def get_title_and_link(self, container):
        tag = container.find('a', class_='ipc-title-link-wrapper')
        if tag:
            name = tag.find('h3').text.strip() 
            if ". " in name:
                name = name.split(". ", 1)[1]
            link = "https://www.imdb.com" + tag.get('href').split('?')[0]
            return name, link
        return "N/A", "N/A"
    
    def get_metadata(self, container):
        """Extracts Year and Duration by searching all spans for specific patterns."""
        all_spans = container.find_all('span')
        
        year = "N/A"
        duration = "N/A"

        for span in all_spans:
            text = span.get_text(strip=True)
            
            if text.isdigit() and len(text) == 4:
                year = text
            
            elif ('h' in text or 'm' in text) and any(char.isdigit() for char in text):
                if len(text) < 10: # Durations are short
                    duration = text
                    
        return year, duration
    
    def get_metascore(self, container):
        score_tag = container.find('span', class_='metacritic-score-box')
        if not score_tag:
            score_tag = container.find('span', style=lambda s: s and 'background-color' in s)
        return score_tag.text.strip() if score_tag else "N/A" 
    
    def get_storyline(self, container):
        story_tag = container.find('div', class_='ipc-html-content-inner-div')
        return story_tag.text.strip() if story_tag else "N/A"

    def get_ratings(self, container):
        rating_tag = container.find('span', class_='ipc-rating-star--rating')
        rating = rating_tag.text.strip() if rating_tag else "N/A"
        
        votes_tag = container.find('span', class_='ipc-rating-star--voteCount')
        votes = votes_tag.text.strip().replace('(', '').replace(')', '').strip() if votes_tag else "N/A"
        return rating, votes

    def scrape_range(self, browser, start, end):
        """Scrapes only the movies within the [start:end] range."""
        page_soup = soup(browser.html, 'html.parser')
        all_containers = page_soup.find_all('div', class_='ipc-metadata-list-summary-item__c')
        
        target_containers = all_containers[start:end]

        for item in target_containers:
            name, link         = self.get_title_and_link(item)
            year, duration     = self.get_metadata(item)
            metascore          = self.get_metascore(item)
            story              = self.get_storyline(item)
            imdb_rating, votes = self.get_ratings(item)

            self.results.append({
                "movie_name"  : name,
                "year"        : year,
                "duration"    : duration,
                "imdb_rating" : imdb_rating,
                "vote_count"  : votes,
                "metascore"   : metascore,
                "storyline"   : story,
                "url"         : link
            })
                    
    def start_scraping(self, start=0, end=100): 
        browser = self._init_browser()
        browser.visit(self.base_url)
        time.sleep(5)
        
        while True:
            temp_soup = soup(browser.html, 'html.parser')
            current_count = len(temp_soup.find_all('div', class_='ipc-metadata-list-summary-item__c'))
            
            if current_count >= end:
                break
            
            try:
                browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1)
                load_more = browser.find_by_text('50 more')
                if load_more:
                    load_more.click()
                    time.sleep(random.uniform(3, 5))
                else:
                    print("🏁 Reached end of IMDb list before reaching your 'end' number.")
                    break
            except:
                break
        self.scrape_range(browser, start, end)

        browser.quit()
        self.save_to_csv()
    
    def save_to_csv(self):
        os.makedirs(r'..\data\raw', exist_ok=True)
        output_path = r'..\data\raw\imdb_movies.csv'
        
        df = pd.DataFrame(self.results) 
        file_exists = os.path.isfile(output_path)
        
        df.to_csv(output_path, mode='a', index=False, header=not file_exists)
        
        print(f"\nData appended! Total rows added in this session: {len(df)}")
        print(f"📍 File location: {output_path}")
        
        

In [ ]:
current_start = 1000 

for i in range(10):
    URL = "https://www.imdb.com/search/title/?title_type=feature"
    scraper = MovieScraper(URL)
    current_end = current_start + 1000
    
    print(f"\n--- 🛰️ Starting Batch {i+1}: {current_start} to {current_end} ---")
    
    try:
        scraper.start_scraping(start=current_start, end=current_end)

        current_start = current_end
        
        print(f"✅ Batch {i+1} finished. Resting for 15 seconds...")
        time.sleep(15) 
        
    except Exception as e:
        print(f"❌ Error in Batch {i+1}: {e}")
        print("Waiting 60 seconds to retry...")
        time.sleep(60)

print(f"🏁 All batches completed! Final count reached: {current_start}")


--- 🛰️ Starting Batch 1: 1000 to 2000 ---

Data appended! Total rows added in this session: 0
📍 File location: ..\data\raw\imdb_movies.csv
✅ Batch 1 finished. Resting for 15 seconds...

--- 🛰️ Starting Batch 2: 2000 to 3000 ---

Data appended! Total rows added in this session: 0
📍 File location: ..\data\raw\imdb_movies.csv
✅ Batch 2 finished. Resting for 15 seconds...

--- 🛰️ Starting Batch 3: 3000 to 4000 ---

Data appended! Total rows added in this session: 1000
📍 File location: ..\data\raw\imdb_movies.csv
✅ Batch 3 finished. Resting for 15 seconds...

--- 🛰️ Starting Batch 4: 4000 to 5000 ---
